# AI651 Assignment 2, Question 2: Motion or Memory?

GitHub: https://github.com/YOUR-USERNAME/YOUR-REPO  <!-- replace; repo must be PUBLIC at submission -->

Run top to bottom. `SMOKE = True` uses fake data and tiny splits so you can debug the pipeline in minutes.
Set `SMOKE = False` and restart the kernel for the real run.

**Commit plan:** commit after the Table 1 checks pass, then after Figure 1. Do not run later experiments before committing the Q2.5 and Q2.11 predictions.

## 0. Imports, config, environment

In [1]:
import sys, time, random, math, platform
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

SMOKE = True                    # True: fake data, 1 epoch, tiny splits (debug). False: real run.
USE_FAKE_DATA = SMOKE
EPOCHS = 1 if SMOKE else 30

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
                      else "cpu")
HARDWARE = torch.cuda.get_device_name(0) if device.type == "cuda" else str(device)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

# record the environment (copy into the report's timing note)
print("python", sys.version.split()[0], "| torch", torch.__version__, "| numpy", np.__version__, "|", platform.platform())
print("device:", HARDWARE, "| SMOKE:", SMOKE, "| EPOCHS:", EPOCHS)

python 3.11.13 | torch 2.8.0 | numpy 2.2.6 | macOS-26.6.2-arm64-arm-64bit
device: mps | SMOKE: True | EPOCHS: 1


## 1. Data and manual split by video index

In [2]:
if USE_FAKE_DATA:
    g_ = torch.Generator().manual_seed(0)
    raw = (torch.rand(10000, 20, 1, 64, 64, generator=g_) > 0.97).to(torch.uint8) * 255
else:
    from torchvision.datasets import MovingMNIST
    raw = MovingMNIST(root="./data", split=None, download=True).data   # uint8, all 10,000 videos

if raw.ndim == 4: raw = raw.unsqueeze(2)
assert raw.shape == (10000, 20, 1, 64, 64), raw.shape

train_v, val_v, test_v = raw[:8000], raw[8000:9000], raw[9000:]   # videos 0-7999 / 8000-8999 / 9000-9999
if SMOKE:
    train_v, val_v, test_v = train_v[:256], val_v[:64], test_v[:64]
print(train_v.shape, val_v.shape, test_v.shape, "| dtype", train_v.dtype)

torch.Size([256, 20, 1, 64, 64]) torch.Size([64, 20, 1, 64, 64]) torch.Size([64, 20, 1, 64, 64]) | dtype torch.uint8


## 2. Transforms, symmetry group, windows

In [3]:
def M(x):            # mirror: x[i,j] -> x[i,63-j]
    return torch.flip(x, dims=(-1,))

def R(x, k=1):       # 90 deg anticlockwise, k times
    return torch.rot90(x, k=k, dims=(-2, -1))

G = [(m, a) for m in (0, 1) for a in range(4)]   # (mirror?, a): g(x) = M(R^a(x)) if mirror else R^a(x)

def g_apply(x, g):
    m, a = g
    y = R(x, a)
    return M(y) if m else y

def g_inv(x, g):
    m, a = g
    y = M(x) if m else x
    return R(y, -a)

def window(video, k=1, t=1):
    """video (20,1,64,64); t is the 1-based start frame; k is the speed.
    Returns X (5,...), Y (5,...) zero-padded where the target frame > 20, and n_valid."""
    in_idx = [t - 1 + k * i for i in range(5)]
    tg_idx = [t - 1 + k * (5 + i) for i in range(5)]
    assert in_idx[-1] <= 19, "input frames out of range"
    valid = [j for j in tg_idx if j <= 19]
    X = video[in_idx]
    Y = video.new_zeros((5,) + tuple(video.shape[1:]))
    if valid: Y[:len(valid)] = video[valid]
    return X, Y, len(valid)

def allowed_starts(k, mode="speed"):
    if mode == "normal":                       # models A, B: all 5 targets exist
        assert k == 1; return list(range(1, 12))
    return list(range(1, 20 - 6 * k + 1))      # models C, D: 5 inputs and >= 2 targets

print({k: len(allowed_starts(k)) for k in (1, 2, 3)}, "starts (expect 14, 8, 2)")

{1: 14, 2: 8, 3: 2} starts (expect 14, 8, 2)


## 3. Table 1: sanity checks (everything must match before going on)

In [4]:
torch.manual_seed(0); x = torch.rand(20, 1, 64, 64)
p = torch.zeros(20, 1, 64, 64); p[7, 0, 10, 20] = 1

def bright(t):
    i = (t > 0.5).nonzero()[0]
    return (int(i[0]), int(i[2]), int(i[3]))           # (frame, row, col)

fr = torch.arange(1, 21).float().view(20, 1, 1, 1).expand(20, 1, 64, 64).clone()
def frames(k):
    X, Y, n = window(fr, k, 1)
    return X[:, 0, 0, 0].int().tolist(), Y[:n, 0, 0, 0].int().tolist()

checks = {
    "(a) max|M(M(x))-x|":        ((M(M(x)) - x).abs().max().item(), torch.equal(M(M(x)), x)),
    "(b) max|R4(x)-x|":          ((R(x, 4) - x).abs().max().item(), torch.equal(R(x, 4), x)),
    "(c) max|MRM(x)-R^-1(x)|":   ((M(R(M(x))) - R(x, -1)).abs().max().item(), torch.equal(M(R(M(x))), R(x, -1))),
}
for name, (d, eq) in checks.items():
    print(name, "=", d, "| torch.equal:", eq)

yours = {
    "(d) bright pixel of M(p)": (bright(M(p)), (7, 10, 43)),
    "(e) bright pixel of R(p)": (bright(R(p)), (7, 43, 10)),
    "(f) 1x input/target":      (frames(1), ([1, 2, 3, 4, 5], [6, 7, 8, 9, 10])),
    "(g) 2x input/target":      (frames(2), ([1, 3, 5, 7, 9], [11, 13, 15, 17, 19])),
    "(h) 3x input/target":      (frames(3), ([1, 4, 7, 10, 13], [16, 19])),
}
ok = all(d == 0 and eq for d, eq in checks.values())
for name, (got, exp) in yours.items():
    print(name, "| yours:", got, "| expected:", exp, "| OK" if got == exp else "| MISMATCH")
    ok &= (got == exp)
print("\nALL TABLE 1 CHECKS PASSED" if ok else "\nFIX THE CODE BEFORE CONTINUING")
assert ok

(a) max|M(M(x))-x| = 0.0 | torch.equal: True
(b) max|R4(x)-x| = 0.0 | torch.equal: True
(c) max|MRM(x)-R^-1(x)| = 0.0 | torch.equal: True
(d) bright pixel of M(p) | yours: (7, 10, 43) | expected: (7, 10, 43) | OK
(e) bright pixel of R(p) | yours: (7, 43, 10) | expected: (7, 43, 10) | OK
(f) 1x input/target | yours: ([1, 2, 3, 4, 5], [6, 7, 8, 9, 10]) | expected: ([1, 2, 3, 4, 5], [6, 7, 8, 9, 10]) | OK
(g) 2x input/target | yours: ([1, 3, 5, 7, 9], [11, 13, 15, 17, 19]) | expected: ([1, 3, 5, 7, 9], [11, 13, 15, 17, 19]) | OK
(h) 3x input/target | yours: ([1, 4, 7, 10, 13], [16, 19]) | expected: ([1, 4, 7, 10, 13], [16, 19]) | OK

ALL TABLE 1 CHECKS PASSED


## 4. Figure 1: optical flow on the first 50 training videos

In [ ]:
import cv2
thetas, speeds = [], []
for v in train_v[:50]:
    f8 = [np.ascontiguousarray(v[s, 0].numpy()) for s in range(20)]      # uint8 frames
    for s in range(19):
        flow = cv2.calcOpticalFlowFarneback(f8[s], f8[s + 1], None, 0.5, 3, 15, 3, 5, 1.2, 0)
        dx, dy = flow[..., 0], flow[..., 1]
        mag = np.sqrt(dx ** 2 + dy ** 2)
        keep = (f8[s] > 51) & (f8[s + 1] > 51) & (mag > 0.5)             # 51 = 0.2 * 255
        thetas.append(np.degrees(np.arctan2(-dy[keep], dx[keep])) % 360)
        speeds.append(mag[keep])
thetas, speeds = np.concatenate(thetas), np.concatenate(speeds)
med = np.median(speeds) if len(speeds) else float("nan")
print("pixels kept:", len(speeds), "| median speed (px/frame):", med)   # put both in the caption

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
cnt, edges = np.histogram(thetas, bins=18, range=(0, 360))
ax[0].bar(edges[:-1], cnt / max(cnt.sum(), 1), width=20, align="edge", edgecolor="k")
ax[0].axhline(1 / 18, color="r", ls="--")
ax[0].set_xlabel("flow direction (deg; 0 = right, 90 = up)"); ax[0].set_ylabel("fraction of kept pixels")
ax[1].hist(speeds, bins=40, weights=np.ones_like(speeds) / max(len(speeds), 1), edgecolor="k")
ax[1].axvline(med, color="r"); ax[1].set_xlabel("flow speed (px/frame)"); ax[1].set_ylabel("fraction of kept pixels")
plt.tight_layout()
plt.savefig("results_fig1_optical_flow.png", dpi=200)     # move into q2/results/ when committing
plt.show()